# Notebook 3: RAG Pipeline — Zillow Data Retrieval
## Iowa City Housing LLM

**Prerequisites:** Complete `01_environment_setup.ipynb` and `02_finetune_llama3.ipynb`.

This notebook builds the **Retrieval-Augmented Generation (RAG)** layer that:
1. Embeds your Zillow JSONL documents into a FAISS vector index
2. At query time, retrieves the most relevant chunks from the index
3. Injects retrieved context into the prompt before generating a response
4. Saves the vector index for reuse

---
### Why RAG on top of fine-tuning?
- Fine-tuning bakes in Iowa City housing **knowledge and conversational style**
- RAG provides **current, updatable data** (today's Zillow prices, new listings, etc.)
- As new Zillow data arrives, you only re-index — no re-training required

## 1. Imports & Config

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [2]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

RAG_CONFIG = {
    "raw_rag_data": "../MarketMindAI/housing_rag_zillow.jsonl",
    "rag_index_path": "../models/faiss_rag_index",
    "finetuned_model_path": "SiwgiLi/llama3-housing-lora-merged",
    "embedding_model": "openai/clip-vit-base-patch32",
    "chunk_size": 512,
    "chunk_overlap": 64,
    "top_k_docs": 3,
    "similarity_threshold": 0.3,
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.9,
}

os.makedirs(RAG_CONFIG["rag_index_path"], exist_ok=True)
print("Config ready")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Config ready


## 2. Load & Chunk the Zillow JSONL Data

In [3]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_rag_documents(jsonl_path):
    """Load Zillow JSONL and convert to LangChain Documents."""
    documents = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            metadata = {
                "id": record.get("id", ""),
                "source": record.get("source_name", ""),
                "url": record.get("source_url", ""),
                "date": record.get("publication_date", ""),
                "section": record.get("section", ""),
                "tags": ", ".join(record.get("topic_tags", [])),
            }
            doc = Document(page_content=record["text"], metadata=metadata)
            documents.append(doc)
    return documents

raw_docs = load_rag_documents(RAG_CONFIG["raw_rag_data"])
print(f"Loaded {len(raw_docs)} raw documents")

for doc in raw_docs:
    print(f"  [{doc.metadata['id']}] {doc.metadata['section']} ({len(doc.page_content)} chars)")

Loaded 3 raw documents
  [housing_zillow_001] Zillow Home Value Index (ZHVI) — Citywide Overview (993 chars)
  [housing_zillow_002] Zillow Neighborhood Breakdown — Iowa City (743 chars)
  [housing_zillow_003] Zillow Observed Rent Index (ZORI) — Iowa City (866 chars)


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CONFIG["chunk_size"],
    chunk_overlap=RAG_CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", " ", ""],
)

split_docs = text_splitter.split_documents(raw_docs)

print(f"Split into {len(split_docs)} chunks from {len(raw_docs)} documents")
lengths = [len(doc.page_content) for doc in split_docs]
print(f"Chunk lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}")

print("\nSample chunk:")
print(split_docs[0].page_content[:300])

Split into 8 chunks from 3 documents
Chunk lengths — min: 178, max: 460, avg: 324

Sample chunk:
As of February 28, 2026, the Zillow Home Value Index (ZHVI) for Iowa City, IA is $292,103,
representing a 4.6 percent increase over the prior year. The ZHVI is built from monthly
changes in property-level Zestimates and is designed to capture both the level and
directional change of home values acro


## 3. Create Embeddings & Build FAISS Index

In [9]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load CLIP for Multimodal Embeddings
clip_model_id = "openai/clip-vit-base-patch32"
print(f"Loading multimodal embedding model: {clip_model_id}")

clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id, use_safetensors=True).to(device)

def get_embedding(text=None, image_path=None):
    """Generates a 512-dimensional vector for either text or an image."""
    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            embedding = clip_model.get_image_features(**inputs)
    elif text:
        inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            embedding = clip_model.get_text_features(**inputs)
    else:
        raise ValueError("Must provide either text or image_path")
        
    # Safely extract the tensor if Transformers v5 returns an object
    if not isinstance(embedding, torch.Tensor):
        if hasattr(embedding, "text_embeds") and embedding.text_embeds is not None:
            embedding = embedding.text_embeds
        elif hasattr(embedding, "image_embeds") and embedding.image_embeds is not None:
            embedding = embedding.image_embeds
        elif hasattr(embedding, "pooler_output"):
            embedding = embedding.pooler_output
        else:
            embedding = embedding[1] if len(embedding) > 1 else embedding[0]
            
    embedding = embedding / embedding.norm(p=2, dim=-1, keepdim=True)
    return embedding.cpu().numpy()[0]

# Test it!
test_embedding = get_embedding(text="Iowa City home prices")
print(f"Multimodal CLIP model loaded! Dimension: {len(test_embedding)}")

Loading multimodal embedding model: openai/clip-vit-base-patch32


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Multimodal CLIP model loaded! Dimension: 512


In [10]:

import faiss
import numpy as np
import json
import os

print("Building multimodal FAISS index...")

# 1. Initialize empty FAISS index for CLIP (512 dimensions, Inner Product for cosine sim)
dimension = 512
index = faiss.IndexFlatIP(dimension)
metadata_store = {}
vector_id = 0

# 2. Add Text Chunks (from your existing split_docs)
print(f"Adding {len(split_docs)} text chunks...")
for doc in split_docs:
    # Get the embedding using our new CLIP function
    vector = get_embedding(text=doc.page_content)
    
    # FAISS requires float32 numpy arrays
    index.add(np.array([vector], dtype=np.float32))
    
    # Store metadata so we can retrieve the text later
    metadata_store[vector_id] = {
        "type": "text",
        "content": doc.page_content,
        "source": doc.metadata.get("source", "Unknown"),
        "section": doc.metadata.get("section", ""),
        "date": doc.metadata.get("date", "")
    }
    vector_id += 1

# 3. Add Image Data (Placeholder - add your RapidAPI or Assessor images here)
# image_paths = ["./data/images/brick_house_52245.jpg", "./data/images/campus_condo_52242.jpg"]
# print(f"Adding images...")
# for img_path in image_paths:
#     vector = get_embedding(image_path=img_path)
#     index.add(np.array([vector], dtype=np.float32))
#     metadata_store[vector_id] = {
#         "type": "image",
#         "file_path": img_path
#     }
#     vector_id += 1

# 4. Save both the FAISS index and the Metadata dictionary
os.makedirs(RAG_CONFIG["rag_index_path"], exist_ok=True)

index_file = os.path.join(RAG_CONFIG["rag_index_path"], "multimodal_index.faiss")
faiss.write_index(index, index_file)

meta_file = os.path.join(RAG_CONFIG["rag_index_path"], "metadata.json")
with open(meta_file, "w") as f:
    json.dump(metadata_store, f)

print(f"✓ FAISS index saved to: {index_file}")
print(f"✓ Metadata saved to: {meta_file}")
print(f"Total vectors: {index.ntotal}")

Building multimodal FAISS index...
Adding 8 text chunks...
✓ FAISS index saved to: ../models/faiss_rag_index\multimodal_index.faiss
✓ Metadata saved to: ../models/faiss_rag_index\metadata.json
Total vectors: 8


## 4. Test the Retrieval System

In [11]:
import numpy as np

def retrieve_context(query_text=None, query_image_path=None, top_k=3):
    """
    Retrieves the most relevant text or images from the multimodal FAISS index.
    Handles text-only, image-only, or combined text+image queries.
    """
    # 1. Determine what the user provided and get the vector
    if query_text and query_image_path:
        # User provided BOTH: Get both vectors and average them
        text_vec = get_embedding(text=query_text)
        img_vec = get_embedding(image_path=query_image_path)
        combined_vec = (text_vec + img_vec) / 2.0
        # Re-normalize the combined vector for cosine similarity
        query_vector = combined_vec / np.linalg.norm(combined_vec)
        
    elif query_text:
        # User provided TEXT only
        query_vector = get_embedding(text=query_text)
        
    elif query_image_path:
        # User provided IMAGE only
        query_vector = get_embedding(image_path=query_image_path)
        
    else:
        return [] # Nothing to search

    # 2. Search the raw FAISS index
    # Note: index and metadata_store must be loaded in memory
    distances, indices = index.search(np.array([query_vector], dtype=np.float32), top_k)

    # 3. Format the results using our metadata dictionary
    retrieved = []
    for i, idx in enumerate(indices[0]):
        if idx == -1: 
            continue # FAISS returns -1 if it runs out of vectors
            
        match_data = metadata_store[idx]
        retrieved.append({
            "score": distances[0][i],
            "type": match_data["type"],
            "content": match_data.get("content", ""), # Text if it's a text chunk
            "file_path": match_data.get("file_path", ""), # Path if it's an image
            "source": match_data.get("source", "Unknown"),
            "section": match_data.get("section", "")
        })
        
    return retrieved

# --- Test it out! ---
print("Testing Text Query:")
results = retrieve_context(query_text="What is the average rent in Iowa City?")
for r in results:
    if r['type'] == 'text':
        print(f"[{r['score']:.3f}] TEXT: {r['content'][:100]}...")
    else:
        print(f"[{r['score']:.3f}] IMAGE: {r['file_path']}")

Testing Text Query:
[0.808] TEXT: Iowa City average rents are approximately 31 percent below the national average, reflecting
both Iow...
[0.807] TEXT: Iowa City Rental Market (Zillow Observed Rent Index / ZORI, as of February 28, 2026):

 Average Rent...
[0.760] TEXT: As of February 28, 2026, the Zillow Home Value Index (ZHVI) for Iowa City, IA is $292,103,
represent...


In [12]:
user_query = "Show me a brick starter home"

# Embed the user's text question
query_vector = get_embedding(text=user_query)

# Search FAISS for the top 2 closest matches
k = 2
distances, indices = index.search(np.array([query_vector]), k)

print("Retrieved Context:")
for i, idx in enumerate(indices[0]):
    match_data = metadata_store[idx]
    score = distances[0][i]
    
    if match_data["type"] == "text":
        print(f"[{score:.3f}] TEXT: {match_data['content']}")
    elif match_data["type"] == "image":
        print(f"[{score:.3f}] IMAGE FOUND: Sending {match_data['file_path']} to Gradio UI")

Retrieved Context:
[0.740] TEXT: The 0.983 sale-to-list ratio indicates buyers are on average negotiating approximately
1.7 percent off asking price — a modestly buyer-favorable dynamic compared to the
competitive conditions of 2021–2022.
[0.687] TEXT: Typical Home Value (ZHVI): $292,103 (+4.6% year-over-year)
 Median Sale Price: $289,417 (as of January 31, 2026)
 Median List Price: $297,567
 Median Sale-to-List Ratio: 0.983 (98.3%)
 For-Sale Inventory: 248 active listings
 New Listings: 54 (February 28, 2026)
 Percent of Sales Over List: 10.2%
 Percent of Sales Under List: 68.1%
 Median Days to Pending: 56 days


## 5. Build the Full RAG + LLM Pipeline

In [15]:
import os
import json
import faiss
import numpy as np
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

SYSTEM_PROMPT_RAG = """You are an Iowa City Housing Assistant specializing in ZIP codes 
52245 and 52242. You help recent college graduates make informed housing decisions and avoid financial scams.

You will be given CONTEXT (text and images) from the latest local data. Use this context 
to provide specific answers. If an image is provided, analyze it carefully. Always clarify 
that your guidance is educational and not professional financial advice."""

def format_rag_text_context(retrieved_docs):
    """Extracts only the text chunks to build the reading context."""
    text_docs = [doc for doc in retrieved_docs if doc['type'] == 'text']
    
    if not text_docs:
        return "No specific current text data found. Providing general guidance."
        
    context_parts = []
    for i, doc in enumerate(text_docs):
        context_parts.append(
            f"[Source {i+1}: {doc.get('source', 'Unknown')}, {doc.get('date', 'Unknown')}]\n{doc['content']}"
        )
    return "\n\n".join(context_parts)


class IowaCityHousingVLM_RAG:
    """Full Multimodal RAG pipeline with Qwen2-VL Vision-Language Model."""

    def __init__(self, vlm_model_id, rag_index_path):
        print("Loading Multimodal RAG components...")
        
        # 1. Load the raw FAISS index & Metadata
        index_file = os.path.join(rag_index_path, "multimodal_index.faiss")
        self.index = faiss.read_index(index_file)
        
        meta_file = os.path.join(rag_index_path, "metadata.json")
        with open(meta_file, "r") as f:
            self.metadata_store = {int(k): v for k, v in json.load(f).items()}

        # 2. Load the Qwen2-VL Model and Processor
        print(f"Loading Vision-Language Model: {vlm_model_id}")
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            vlm_model_id,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(vlm_model_id)
        print("Multimodal VLM RAG pipeline ready!")

    def retrieve_context(self, query_text=None, query_image_path=None, top_k=3):
        """Searches FAISS using text, image, or both."""
        if query_text and query_image_path:
            text_vec = get_embedding(text=query_text)
            img_vec = get_embedding(image_path=query_image_path)
            combined_vec = (text_vec + img_vec) / 2.0
            query_vector = combined_vec / np.linalg.norm(combined_vec)
        elif query_text:
            query_vector = get_embedding(text=query_text)
        elif query_image_path:
            query_vector = get_embedding(image_path=query_image_path)
        else:
            return []

        distances, indices = self.index.search(np.array([query_vector], dtype=np.float32), top_k)

        retrieved = []
        for i, idx in enumerate(indices[0]):
            if idx == -1: continue
            match_data = self.metadata_store[idx]
            retrieved.append({
                "score": distances[0][i],
                "type": match_data["type"],
                "content": match_data.get("content", ""),
                "file_path": match_data.get("file_path", ""),
                "source": match_data.get("source", "Unknown"),
                "section": match_data.get("section", ""),
            })
        return retrieved

    # ADDED verbose=False here!
    def answer(self, question_text=None, question_image_path=None, top_k=3, max_new_tokens=512, verbose=False):
        # 1. Retrieve multimodal documents
        retrieved_docs = self.retrieve_context(question_text, question_image_path, top_k)
        
        # ADDED the verbose printing block back in!
        if verbose:
            print(f"Retrieved {len(retrieved_docs)} chunks")
            for doc in retrieved_docs:
                item = doc['section'] if doc['type'] == 'text' else doc['file_path']
                print(f"  [{doc['score']:.3f}] {doc['type'].upper()}: {item}")
        
        # 2. Extract context
        context_text = format_rag_text_context(retrieved_docs)
        retrieved_images = [doc['file_path'] for doc in retrieved_docs if doc['type'] == 'image']
        
        # 3. Build the VLM Payload
        user_content = []
        
        if question_image_path:
            user_content.append({"type": "image", "image": question_image_path})
            
        for img_path in retrieved_images:
            user_content.append({"type": "image", "image": img_path})
            
        prompt_question = question_text if question_text else "Please analyze the provided image(s)."
        full_text_prompt = f"CONTEXT (from Zillow data):\n{context_text}\n\nUSER QUESTION: {prompt_question}"
        user_content.append({"type": "text", "text": full_text_prompt})

        # 4. Construct the standard Chat Template
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_RAG}]},
            {"role": "user", "content": user_content}
        ]

        # 5. Process and Generate (Wrapped in no_grad for memory safety!)
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        with torch.no_grad():
            inputs = self.processor(
                text=[text_prompt],
                images=image_inputs,
                padding=True,
                return_tensors="pt"
            ).to(self.model.device)

            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        response = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        
        return {
            "question": prompt_question,
            "answer": response,
            "retrieved_images": retrieved_images,
            "sources": [{"type": d["type"], "score": d["score"]} for d in retrieved_docs]
        }

# Initialize the new VLM pipeline
vlm_model_id = "Qwen/Qwen2-VL-2B-Instruct"

rag_pipeline = IowaCityHousingVLM_RAG(
    vlm_model_id=vlm_model_id,
    rag_index_path=RAG_CONFIG["rag_index_path"]
)

Loading Multimodal RAG components...
Loading Vision-Language Model: Qwen/Qwen2-VL-2B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Multimodal VLM RAG pipeline ready!


## 6. Test the Full RAG Pipeline

In [16]:
rag_test_questions = [
    "What is the current average rent in Iowa City?",
    "How much have home values increased in Iowa City this year?",
    "Which Iowa City neighborhood should I look at as a first-time buyer under $250,000?",
    "Is it currently a buyer's or seller's market in Iowa City?",
]

for question in rag_test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('='*60)
    result = rag_pipeline.answer(question, verbose=True)
    print(f"\nA: {result['answer']}")
    print(f"\nSources: {result['sources']}")


Q: What is the current average rent in Iowa City?
Retrieved 3 chunks
  [0.832] TEXT: Zillow Observed Rent Index (ZORI) — Iowa City
  [0.814] TEXT: Zillow Observed Rent Index (ZORI) — Iowa City
  [0.782] TEXT: Zillow Home Value Index (ZHVI) — Citywide Overview

A: The current average rent in Iowa City is $1,308 per month.

Sources: [{'type': 'text', 'score': 0.8316231}, {'type': 'text', 'score': 0.8141701}, {'type': 'text', 'score': 0.78237355}]

Q: How much have home values increased in Iowa City this year?
Retrieved 3 chunks
  [0.826] TEXT: Zillow Home Value Index (ZHVI) — Citywide Overview
  [0.823] TEXT: Zillow Neighborhood Breakdown — Iowa City
  [0.812] TEXT: Zillow Observed Rent Index (ZORI) — Iowa City

A: The Zillow Home Value Index (ZHVI) for Iowa City, IA has increased by 4.6 percent over the prior year.

Sources: [{'type': 'text', 'score': 0.8258953}, {'type': 'text', 'score': 0.8227445}, {'type': 'text', 'score': 0.81166315}]

Q: Which Iowa City neighborhood should I look 

## 7. Updating the Index with New Data

In [17]:
import os
import json
import faiss
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter

def add_new_data_to_index(rag_index_path, new_jsonl_path=None, new_image_folder=None):
    """
    Add new monthly Zillow text data and/or property images to the multimodal FAISS index.
    No model re-training needed — just re-index!
    """
    print("Loading existing multimodal index and metadata...")
    
    # 1. Load existing index and metadata
    index_file = os.path.join(rag_index_path, "multimodal_index.faiss")
    meta_file = os.path.join(rag_index_path, "metadata.json")
    
    index = faiss.read_index(index_file)
    with open(meta_file, "r") as f:
        # Convert JSON string keys back to integers for FAISS
        metadata_store = {int(k): v for k, v in json.load(f).items()}
        
    vector_id = index.ntotal # Next available ID
    vectors_added = 0
    
    # 2. Process and add new Text Data (Zillow JSONL)
    if new_jsonl_path and os.path.exists(new_jsonl_path):
        print(f"Processing new text data from {new_jsonl_path}...")
        new_docs = load_rag_documents(new_jsonl_path) # Uses your function from earlier in Notebook 3
        splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
        new_chunks = splitter.split_documents(new_docs)
        
        for doc in new_chunks:
            vector = get_embedding(text=doc.page_content)
            index.add(np.array([vector], dtype=np.float32))
            metadata_store[vector_id] = {
                "type": "text",
                "content": doc.page_content,
                "source": doc.metadata.get("source", "Unknown"),
                "section": doc.metadata.get("section", ""),
                "date": doc.metadata.get("date", "")
            }
            vector_id += 1
            vectors_added += 1

    # 3. Process and add new Image Data
    if new_image_folder:
        print(f"Adding images from {new_image_folder}...")
        if os.path.exists(new_image_folder):
            valid_exts = {".jpg", ".jpeg", ".png"}
            for filename in os.listdir(new_image_folder):
                if os.path.splitext(filename)[1].lower() in valid_exts:
                    img_path = os.path.join(new_image_folder, filename)
                    vector = get_embedding(image_path=img_path)
                    index.add(np.array([vector], dtype=np.float32))
                    metadata_store[vector_id] = {
                        "type": "image",
                        "file_path": img_path
                    }
                    vector_id += 1
                    vectors_added += 1
        else:
            print(f"Folder {new_image_folder} not found, skipping image addition.")
                
    # 4. Save everything back to disk
    faiss.write_index(index, index_file)
    with open(meta_file, "w") as f:
        json.dump(metadata_store, f)
        
    print(f"✓ Added {vectors_added} new items. Total vectors is now: {index.ntotal}")
    return index, metadata_store

# USAGE — run monthly with new data:
# updated_index, updated_meta = add_new_data_to_index(
#     rag_index_path=RAG_CONFIG["rag_index_path"],
#     new_jsonl_path="../data/housing_rag_zillow_march2026.jsonl",
#     new_image_folder="../data/images_march2026"
# )

## 8. ZIP Code Expansion Roadmap

In [14]:
EXPANSION_PLAN = {
    "Phase 1 (Active)": {"zip_codes": ["52245", "52242"], "note": "Iowa City core — current scope"},
    "Phase 2 (Planned)": {"zip_codes": ["52240", "52246"], "note": "Full Iowa City — add data + re-index only"},
    "Phase 3 (Future)": {"zip_codes": ["52241", "52302"], "note": "Coralville + Marion — may require new fine-tuning"},
    "Phase 4 (Long-term)": {"zip_codes": ["Iowa statewide"], "note": "All major Iowa metros — city-specific LoRA adapters"},
}

for phase, info in EXPANSION_PLAN.items():
    print(f"{phase}: {info['zip_codes']}")
    print(f"  -> {info['note']}")

Phase 1 (Active): ['52245', '52242']
  -> Iowa City core — current scope
Phase 2 (Planned): ['52240', '52246']
  -> Full Iowa City — add data + re-index only
Phase 3 (Future): ['52241', '52302']
  -> Coralville + Marion — may require new fine-tuning
Phase 4 (Long-term): ['Iowa statewide']
  -> All major Iowa metros — city-specific LoRA adapters


## Summary

The RAG pipeline is complete:
- Zillow JSONL documents embedded and indexed in FAISS
- Retrieval tested and working with relevance scoring
- Full `IowaCityHousingRAG` pipeline assembled (retrieve + generate)
- `add_new_data_to_index()` ready for monthly Zillow data updates
- Expansion roadmap documented

**Next:** Run `04_local_testing.ipynb` for the interactive chat UI and full evaluation.